# Notebook 2: out-of-fold evaluation and model comparison

Adda et al. (2022), Modeling and optimization of small-scale NF/RO seawater desalination using the ANN,
*Environ. Eng. Res.* 27(2): 200383.

Notebook 1 reproduces the paper's Table 2, but those numbers are in-sample. Here the same network and
eight other models are scored out-of-fold with repeated cross-validation. We also add prediction
intervals and a cross-validated sensitivity analysis.

Run time is a few minutes (folds run in parallel). From this folder:

```bash
jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=1800 NF_RO_Improved.ipynb
```

In [1]:
import time, warnings, numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import least_squares
from joblib import Parallel, delayed
from sklearn.model_selection import RepeatedKFold, KFold, GridSearchCV, train_test_split
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.multioutput import MultiOutputRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor
warnings.filterwarnings("ignore")

INPUTS  = ["Time", "Pressure", "Temperature", "FeedFlow", "ConductivityFeed"]
OUTPUTS = ["PermeateConductivity", "PermeateFlowRate", "PermeateRecovery"]
UNITS   = {"PermeateConductivity": "uS/cm", "PermeateFlowRate": "m3/h", "PermeateRecovery": "%"}
SEED, N_SPLITS, N_REPEATS = 42, 10, 5
ANN_RESTARTS, ANN_LAMBDA  = 4, 3e-2            # see scripts/ann_grid.py

raw = pd.read_excel("Data_Desalination.xlsx", header=None)
df = raw.iloc[5:, 3:].copy()
df.columns = ["sn", "Time", "Pressure", "Temperature", "FeedFlow", "ConductivityFeed",
              "PermeateRecovery", "PermeateFlowRate", "PermeateConductivity"]
df = df.reset_index(drop=True).drop(columns="sn").astype(float)      # s/n is a row counter
df.loc[np.isclose(df["Time"], 852.493), "Temperature"] = 29.2915    # row 45, see Notebook 1

assert len(df) == 73
X_all, Y_all = df[INPUTS].values, df[OUTPUTS].values
print(f"{len(df)} experiments, {len(INPUTS)} inputs -> {len(OUTPUTS)} outputs")
print(f"protocol: RepeatedKFold({N_SPLITS} splits x {N_REPEATS} repeats) = {N_SPLITS*N_REPEATS} folds")

73 experiments, 5 inputs -> 3 outputs
protocol: RepeatedKFold(10 splits x 5 repeats) = 50 folds


## 1. Differences from the paper

| | Paper | This notebook |
|---|---|---|
| Evaluation | in-sample, 60 training rows | 10-fold CV repeated 5 times, out-of-fold |
| Models | 5-8-3 ANN and MLR | the same two plus ridge, a regularised ANN, SVR, Gaussian process, random forest, XGBoost and an RF + XGBoost average |
| Uncertainty | none | Gaussian-process 95 % prediction intervals |
| Sensitivity | Garson's method on one network | permutation importance in every CV fold |

The data are the same as in Notebook 1 (`s/n` dropped, row 45 temperature corrected).

## 2. Evaluation protocol

- `RepeatedKFold(n_splits=10, n_repeats=5)`, 50 fits per model.
- In each repeat the 10 folds cover all 73 rows, giving one out-of-fold prediction per row. Metrics
  are computed on those 73 predictions and averaged over the 5 repeats.
- Metrics are in engineering units: R = `corrcoef(y, ŷ)`, RMSE, MAE and AARD = 100·mean|(y − ŷ)/y|.
- Scalers and hyperparameter searches are fitted inside each training fold. The exception is the
  paper's own recipe (A0 below), which scales on all 73 rows as the paper does.

In [2]:
def unit_metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return dict(R=float(np.corrcoef(y, p)[0, 1]),
                RMSE=float(np.sqrt(np.mean((y - p) ** 2))),
                MAE=float(np.mean(np.abs(y - p))),
                AARD=float(100 * np.mean(np.abs((y - p) / y))))


# 5-8-3 network from the paper, trained with Levenberg-Marquardt
class _ANNCore:
    def __init__(self, seed=0):
        r = np.random.default_rng(seed)
        self.t0 = np.concatenate([r.normal(scale=.3, size=40), r.normal(scale=.3, size=8),
                                  r.normal(scale=.3, size=24), r.normal(scale=.3, size=3)])
    @staticmethod
    def unpack(t):
        return t[:40].reshape(8, 5), t[40:48], t[48:72].reshape(3, 8), t[72:75]
    def forward(self, X, t=None):
        W1, b1, W2, b2 = self.unpack(self.t_ if t is None else t)
        return np.tanh(np.asarray(X) @ W1.T + b1) @ W2.T + b2
    def fit(self, X, Y, lam, max_nfev):
        def resid(t):
            e = (self.forward(X, t) - Y).ravel()
            return e if lam == 0 else np.concatenate([e, np.sqrt(lam) * t])
        r = least_squares(resid, self.t0, method="lm", max_nfev=max_nfev)
        self.t_, self.sse_ = r.x, float(np.sum((self.forward(X, r.x) - Y) ** 2))
        return self


class PaperANN:
    # ext_scale: (xlo, xhi, ylo, yhi) fitted elsewhere, otherwise fitted on the training rows
    # lam: L2 weight decay; ensemble: average the restarts instead of keeping the best
    # max_nfev: LM evaluation cap, used in place of trainlm's early stopping
    def __init__(self, lam=0.0, restarts=ANN_RESTARTS, seed=1, ensemble=False,
                 max_nfev=400, ext_scale=None):
        self.lam, self.restarts, self.seed = lam, restarts, seed
        self.ensemble, self.max_nfev, self.ext_scale = ensemble, max_nfev, ext_scale

    def fit(self, X, Y):
        X, Y = np.asarray(X, float), np.asarray(Y, float)
        if self.ext_scale is None:
            self.xlo, self.xhi, self.ylo, self.yhi = X.min(0), X.max(0), Y.min(0), Y.max(0)
        else:
            self.xlo, self.xhi, self.ylo, self.yhi = self.ext_scale
        self.xhi = np.where(self.xhi > self.xlo, self.xhi, self.xlo + 1)
        self.yhi = np.where(self.yhi > self.ylo, self.yhi, self.ylo + 1)
        Xn = 2 * (X - self.xlo) / (self.xhi - self.xlo) - 1
        Yn = 2 * (Y - self.ylo) / (self.yhi - self.ylo) - 1
        nets = [_ANNCore(self.seed * 100 + s).fit(Xn, Yn, self.lam, self.max_nfev)
                for s in range(self.restarts)]
        self.nets_ = nets if self.ensemble else [min(nets, key=lambda n: n.sse_)]
        return self

    def predict(self, X):
        Xn = 2 * (np.asarray(X, float) - self.xlo) / (self.xhi - self.xlo) - 1
        Pn = np.mean([n.forward(Xn) for n in self.nets_], axis=0)
        return (Pn + 1) * .5 * (self.yhi - self.ylo) + self.ylo


# sklearn models: X and y standardised inside the fold
def ttr(inner):
    return TransformedTargetRegressor(
        regressor=Pipeline([("xs", StandardScaler()), ("est", inner)]),
        transformer=StandardScaler())

N_JOBS = -1     # folds in parallel, inner grid search serial

def gcv(est, grid):
    return GridSearchCV(est, grid, cv=KFold(4, shuffle=True, random_state=0), n_jobs=1)

def make_mlr():   return ttr(LinearRegression())
def make_ridge(): return ttr(RidgeCV(alphas=np.logspace(-3, 3, 25)))

def make_svr():
    inner = MultiOutputRegressor(SVR(kernel="rbf"))
    grid = {"regressor__est__estimator__C": [3, 30, 100],
            "regressor__est__estimator__gamma": [0.05, 0.15],
            "regressor__est__estimator__epsilon": [0.03]}
    return gcv(ttr(inner), grid)

def make_gpr():
    k = (ConstantKernel(1.0, (1e-2, 1e2))
         * Matern(length_scale=1.0, nu=2.5, length_scale_bounds=(1e-2, 1e2))
         + WhiteKernel(1e-1, (1e-4, 1e1)))
    return ttr(GaussianProcessRegressor(kernel=k, normalize_y=False,
                                        n_restarts_optimizer=0, random_state=0))

def make_rf():
    inner = RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=1)
    grid = {"regressor__est__max_depth": [4, None]}
    return gcv(ttr(inner), grid)

def make_xgb():
    inner = MultiOutputRegressor(XGBRegressor(n_estimators=250, learning_rate=0.05,
              max_depth=2, subsample=0.9, colsample_bytree=0.9,
              random_state=0, verbosity=0, n_jobs=1))
    return ttr(inner)

class RFXGB:                                     # average of RF and XGBoost
    def fit(self, X, Y):
        self.a = make_rf().fit(X, Y)
        self.b = make_xgb().fit(X, Y)
        return self
    def predict(self, X):
        return 0.5 * self.a.predict(X) + 0.5 * self.b.predict(X)


# one 73-row out-of-fold prediction per repeat
def _fit_predict_fold(make_est, X, Y, tr, te):
    return te, make_est().fit(X[tr], Y[tr]).predict(X[te])

def oof_by_repeat(make_est, n_splits=N_SPLITS, n_repeats=N_REPEATS, seed=SEED,
                  X=X_all, Y=Y_all, n_jobs=N_JOBS):
    splits = list(RepeatedKFold(n_splits=n_splits, n_repeats=n_repeats,
                                random_state=seed).split(X))
    parts = Parallel(n_jobs=n_jobs)(
        delayed(_fit_predict_fold)(make_est, X, Y, tr, te) for tr, te in splits)
    rows = []
    for rep in range(n_repeats):
        oof = np.full_like(Y, np.nan, dtype=float)
        for k in range(n_splits):
            te, pred = parts[rep * n_splits + k]
            oof[te] = pred
        assert not np.isnan(oof).any()
        for j, o in enumerate(OUTPUTS):
            rows.append(dict(repeat=rep, Output=o, **unit_metrics(Y[:, j], oof[:, j])))
    return pd.DataFrame(rows)


def agg_R(res):
    m = res.groupby("Output").R.mean().reindex(OUTPUTS)
    return dict(zip(OUTPUTS, m.round(3))), round(m.mean(), 3)


# in-sample fit on a 60/13 split, for tables C2 and C3
Xtr_is, Xte_is, Ytr_is, Yte_is = train_test_split(
    X_all, Y_all, test_size=13, random_state=SEED, shuffle=True)

def insample_R(make_est):
    est = make_est().fit(Xtr_is, Ytr_is)
    P = est.predict(Xtr_is)
    return {o: float(np.corrcoef(Ytr_is[:, j], P[:, j])[0, 1]) for j, o in enumerate(OUTPUTS)}

RESULTS, TIMING = {}, {}
def run(name, factory):
    t = time.time(); RESULTS[name] = oof_by_repeat(factory); TIMING[name] = time.time() - t
    per, mean = agg_R(RESULTS[name])
    print(f"{name:34s} {TIMING[name]:6.1f}s   mean R = {mean:.3f}   " +
          "  ".join(f"{o[:4]}={per[o]:.3f}" for o in OUTPUTS))

gx = (X_all.min(0), X_all.max(0), Y_all.min(0), Y_all.max(0))     # scaler on all 73 rows (paper)
print("helpers ready")

helpers ready


## 3. Section A: ablation of the paper's network

Four cumulative changes to the 5-8-3 network:

| step | change |
|---|---|
| A0 | paper's recipe: scaler on all 73 rows, no regularisation, best of 4 restarts, LM capped at 400 evaluations |
| A1 | scaler fitted on the training fold only |
| A2 | L2 weight decay, λ = 0.03 |
| A3 | average the 4 restarts instead of keeping the best one |

A1 lowers R here (0.579 to 0.489): with per-fold scaling some held-out points fall outside [-1, 1]
and the tanh units saturate. The gain comes from A2. λ = 0.03 with H = 8 is the best cell of the grid
in `scripts/ann_grid.py` (λ from 0 to 0.3, H from 4 to 12). That grid is scored on these same folds,
so A2 and A3 are somewhat optimistic.

In [3]:
ablation = {
    "A0  paper method (scaler on all rows)": lambda: PaperANN(lam=0.0, ensemble=False, ext_scale=gx),
    "A1  + scaler on training fold only":    lambda: PaperANN(lam=0.0, ensemble=False),
    "A2  + L2 regularisation":               lambda: PaperANN(lam=ANN_LAMBDA, ensemble=False),
    "A3  + average the 4 restarts":          lambda: PaperANN(lam=ANN_LAMBDA, ensemble=True),
}
abl_rows = []
for name, fac in ablation.items():
    res = oof_by_repeat(fac); per, mean = agg_R(res)
    abl_rows.append({"step": name, **{OUTPUTS[j]: per[OUTPUTS[j]] for j in range(3)}, "MEAN R": mean})
    print(f"{name:38s} mean held-out R = {mean:.3f}")

abl = pd.DataFrame(abl_rows).set_index("step")
abl.to_csv("ablation.csv")
print()
print(abl.round(3).to_string())
print(f"\nregularisation + averaging lift the paper's own network from "
      f"{abl['MEAN R'].iloc[0]:.3f} to {abl['MEAN R'].iloc[-1]:.3f} held-out R.")

A0  paper method (scaler on all rows)  mean held-out R = 0.579


A1  + scaler on training fold only     mean held-out R = 0.489


A2  + L2 regularisation                mean held-out R = 0.750


A3  + average the 4 restarts           mean held-out R = 0.768

                                       PermeateConductivity  PermeateFlowRate  PermeateRecovery  MEAN R
step                                                                                                   
A0  paper method (scaler on all rows)                 0.408             0.643             0.687   0.579
A1  + scaler on training fold only                    0.357             0.526             0.583   0.489
A2  + L2 regularisation                               0.711             0.749             0.791   0.750
A3  + average the 4 restarts                          0.701             0.776             0.828   0.768

regularisation + averaging lift the paper's own network from 0.579 to 0.768 held-out R.


## 4. Section B: model comparison

All models use the protocol above.

- MLR and ridge.
- Adda ANN (published recipe): step A0.
- Improved ANN: step A3.
- SVR with an RBF kernel, C and γ tuned inside each fold.
- Gaussian process, Matérn 5/2 plus white noise.
- Random forest (depth tuned inside each fold), XGBoost (fixed settings, not tuned) and their average.

In [4]:
run("MLR (paper baseline)", make_mlr)
run("Ridge",                 make_ridge)

MLR (paper baseline)                  0.9s   mean R = 0.549   Perm=0.503  Perm=0.640  Perm=0.505
Ridge                                 0.0s   mean R = 0.535   Perm=0.484  Perm=0.633  Perm=0.489


In [5]:
run("Adda ANN (published recipe)", lambda: PaperANN(lam=0.0, ensemble=False, max_nfev=400, ext_scale=gx))
run("Improved ANN (L2 + ensemble)", lambda: PaperANN(lam=ANN_LAMBDA, ensemble=True))

Adda ANN (published recipe)          13.2s   mean R = 0.579   Perm=0.408  Perm=0.643  Perm=0.687


Improved ANN (L2 + ensemble)          4.5s   mean R = 0.768   Perm=0.701  Perm=0.776  Perm=0.828


In [6]:
run("SVR (RBF)",          make_svr)
run("Gaussian process",   make_gpr)

SVR (RBF)                             0.6s   mean R = 0.569   Perm=0.488  Perm=0.652  Perm=0.565
Gaussian process                      0.1s   mean R = 0.645   Perm=0.599  Perm=0.659  Perm=0.677


In [7]:
run("Random forest",      make_rf)
run("XGBoost",            make_xgb)
run("RF + XGB ensemble",  RFXGB)

Random forest                         6.7s   mean R = 0.816   Perm=0.737  Perm=0.818  Perm=0.893


XGBoost                               0.2s   mean R = 0.783   Perm=0.717  Perm=0.786  Perm=0.845


RF + XGB ensemble                     6.7s   mean R = 0.810   Perm=0.738  Perm=0.814  Perm=0.879


In [8]:
def summary_frame(metric, ascending):
    tab = pd.concat({k: v.groupby("Output")[metric].mean().reindex(OUTPUTS)
                     for k, v in RESULTS.items()}, axis=1).T[OUTPUTS]
    tab["MEAN"] = tab.mean(axis=1)
    return tab.sort_values("MEAN", ascending=ascending)

lb_R = summary_frame("R", ascending=False)
lb_R_sd = pd.concat({k: v.groupby("Output").R.std().reindex(OUTPUTS)
                     for k, v in RESULTS.items()}, axis=1).T[OUTPUTS].reindex(lb_R.index)
lb_RMSE = summary_frame("RMSE", ascending=True)
lb_MAE  = summary_frame("MAE",  ascending=True)
lb_AARD = summary_frame("AARD", ascending=True)

lb_R.to_csv("method_leaderboard_R.csv")
lb_RMSE.to_csv("method_leaderboard_RMSE.csv")
lb_MAE.to_csv("method_leaderboard_MAE.csv")
lb_AARD.to_csv("method_leaderboard_AARD.csv")

BEST = lb_R.index[0]
print(f"held-out R, mean over {N_REPEATS} repeats of 10-fold CV\n")
print(lb_R.round(3).to_string())
print()
print(f"best model: {BEST}   (mean held-out R = {lb_R.loc[BEST, 'MEAN']:.3f})")
print(f"paper's ANN as published, same protocol: "
      f"mean held-out R = {lb_R.loc['Adda ANN (published recipe)','MEAN']:.3f}")

held-out R, mean over 5 repeats of 10-fold CV

Output                        PermeateConductivity  PermeateFlowRate  PermeateRecovery   MEAN
Random forest                                0.737             0.818             0.893  0.816
RF + XGB ensemble                            0.738             0.814             0.879  0.810
XGBoost                                      0.717             0.786             0.845  0.783
Improved ANN (L2 + ensemble)                 0.701             0.776             0.828  0.768
Gaussian process                             0.599             0.659             0.677  0.645
Adda ANN (published recipe)                  0.408             0.643             0.687  0.579
SVR (RBF)                                    0.488             0.652             0.565  0.569
MLR (paper baseline)                         0.503             0.640             0.505  0.549
Ridge                                        0.484             0.633             0.489  0.535

best model: 

In [9]:
fig, ax = plt.subplots(figsize=(11, 0.5 + 0.42 * len(lb_R))); ax.axis("off")
disp = lb_R.round(3).copy()
disp.columns = [c.replace("Permeate", "") + ("" if c == "MEAN" else f"\n({UNITS.get(c,'')})")
                for c in disp.columns]
tbl = ax.table(cellText=disp.values, rowLabels=disp.index, colLabels=disp.columns,
               loc="center", cellLoc="center", rowLoc="left")
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 1.6)
for (r, c), cell in tbl.get_celld().items():
    if r == 0 or c == -1:
        cell.set_text_props(fontweight="bold"); cell.set_facecolor("#d9e2ec")
    if r >= 1 and disp.index[r - 1] == BEST:
        cell.set_facecolor("#d7ecd9")
    if r >= 1 and disp.index[r - 1].startswith("Adda ANN"):
        cell.set_facecolor("#f6e0e0")
ax.set_title(f"Held-out R by model  (mean of {N_REPEATS}x 10-fold CV)  -  "
             f"green = best, red = paper's recipe", pad=12)
fig.tight_layout(); fig.savefig("method_leaderboard.png", dpi=150, bbox_inches="tight")
plt.show()

# grouped bar chart for a subset
pick = ["Adda ANN (published recipe)", "MLR (paper baseline)",
        "Improved ANN (L2 + ensemble)", "Gaussian process", "XGBoost", BEST]
pick = list(dict.fromkeys(pick))
fig, ax = plt.subplots(figsize=(11, 4.6))
w = 0.8 / len(pick)
colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(pick)))
for i, m in enumerate(pick):
    ax.bar(np.arange(3) + i * w, [lb_R.loc[m, o] for o in OUTPUTS], w,
           label=m, color=colors[i], edgecolor="k", linewidth=.4)
ax.set_xticks(np.arange(3) + 0.4 - w / 2)
ax.set_xticklabels([o.replace("Permeate", "") for o in OUTPUTS])
ax.set_ylabel("held-out R"); ax.set_ylim(0, 1)
ax.axhline(0.969, ls=":", c="grey"); ax.text(2.35, 0.975, "paper's printed R (in-sample)", fontsize=8, c="grey")
ax.legend(fontsize=8, ncol=2); ax.grid(axis="y", alpha=.3)
ax.set_title("Held-out R by output")
fig.tight_layout(); fig.savefig("leaderboard_bars.png", dpi=150, bbox_inches="tight")
plt.show()

Notes on the leaderboard:

- The paper's network scores mean R = 0.579 out-of-fold. That is 6th of 9 on R and last on the mean
  error metrics. Its printed 0.969 / 0.942 / 0.964 are in-sample.
- This value depends on the LM cap, which stands in for `trainlm` early stopping. Caps of
  3000 / 400 / 100 evaluations give 0.523 / 0.579 / 0.588 (`scripts/lm_cap.py`). We use 400 throughout. The in-sample
  column in section 5 uses 3000.
- The random forest is best (0.737 / 0.818 / 0.893). It was picked on the same folds that report its
  score, so its lead is somewhat optimistic.
- The regularised ANN reaches 0.768 with the same architecture.

## 5. Section C: comparison with the paper

C0 lists the paper's printed R next to the out-of-fold results. C1 compares the random forest with the
paper's recipe under the same protocol. C2 and C3 show the in-sample numbers. No table compares an
in-sample number with an out-of-fold one as an improvement.

In [10]:
# Table C0
PAPER   = "Adda ANN (published recipe)"
IMPR    = "Improved ANN (L2 + ensemble)"
pp      = {"PermeateConductivity": 0.969, "PermeateFlowRate": 0.942, "PermeateRecovery": 0.964}
pann_is0 = insample_R(lambda: PaperANN(lam=0.0, ensemble=False, max_nfev=3000))
Rpaper  = RESULTS[PAPER].groupby("Output").R.mean()
Rbest   = RESULTS[BEST].groupby("Output").R.mean()
Rimpr   = RESULTS[IMPR].groupby("Output").R.mean()
RMSEp   = RESULTS[PAPER].groupby("Output").RMSE.mean()
RMSEb   = RESULTS[BEST].groupby("Output").RMSE.mean()
AARDp   = RESULTS[PAPER].groupby("Output").AARD.mean()
AARDb   = RESULTS[BEST].groupby("Output").AARD.mean()

c0 = []
for o in OUTPUTS:
    c0.append({"Output": o.replace("Permeate", ""),
               "Adda ANN\n(published, in-sample R)": pp[o],
               "Adda ANN\n(held-out R)": round(Rpaper[o], 3),
               "improved ANN\n(held-out R)": round(Rimpr[o], 3),
               f"BEST = {BEST}\n(held-out R)": round(Rbest[o], 3),
               "improvement vs paper ANN\n(held-out)":
                   f"R {Rbest[o]-Rpaper[o]:+.2f} | RMSE {100*(RMSEb[o]/RMSEp[o]-1):+.0f}% | AARD {100*(AARDb[o]/AARDp[o]-1):+.0f}%"})
mean_row = {"Output": "MEAN",
            "Adda ANN\n(published, in-sample R)": round(np.mean(list(pp.values())), 3),
            "Adda ANN\n(held-out R)": round(Rpaper.mean(), 3),
            "improved ANN\n(held-out R)": round(Rimpr.mean(), 3),
            f"BEST = {BEST}\n(held-out R)": round(Rbest.mean(), 3),
            "improvement vs paper ANN\n(held-out)":
                f"R {Rbest.mean()-Rpaper.mean():+.2f} | RMSE {100*(RMSEb.mean()/RMSEp.mean()-1):+.0f}%"}
C0 = pd.DataFrame(c0 + [mean_row]).set_index("Output")
C0.to_csv("improvement_vs_paper.csv")
print("Table C0. Column 1 is the paper's in-sample R; the rest are out-of-fold (10-fold CV x 5).")
print(C0.to_string())

fig, ax = plt.subplots(figsize=(13, 2.4 + 0.1)); ax.axis("off")
tbl = ax.table(cellText=C0.reset_index().values, colLabels=C0.reset_index().columns,
               loc="center", cellLoc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(8.5); tbl.scale(1, 2.4)
tbl.auto_set_column_width(range(len(C0.columns) + 1))
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_text_props(fontweight="bold"); cell.set_facecolor("#d9e2ec")
    elif C0.reset_index().iloc[r - 1, 0] == "MEAN":
        cell.set_text_props(fontweight="bold"); cell.set_facecolor("#eef3f8")
    if c == 4 and r >= 1:
        cell.set_facecolor("#d7ecd9")
    if c == 1 and r >= 1:
        cell.set_facecolor("#f6e0e0")
ax.set_title("Adda et al. (2022) network vs. best model, same out-of-fold protocol", pad=16)
fig.tight_layout(); fig.savefig("improvement_vs_paper.png", dpi=160, bbox_inches="tight")
plt.show()

Table C0. Column 1 is the paper's in-sample R; the rest are out-of-fold (10-fold CV x 5).
              Adda ANN\n(published, in-sample R)  Adda ANN\n(held-out R)  improved ANN\n(held-out R)  BEST = Random forest\n(held-out R) improvement vs paper ANN\n(held-out)
Output                                                                                                                                                                       
Conductivity                               0.969                   0.408                       0.701                               0.737      R +0.33 | RMSE -43% | AARD -38%
FlowRate                                   0.942                   0.643                       0.776                               0.818      R +0.18 | RMSE -40% | AARD -26%
Recovery                                   0.964                   0.687                       0.828                               0.893      R +0.21 | RMSE -48% | AARD -39%
MEAN                                    

In [11]:
# Table C1: best model vs the paper's recipe, same protocol
PAPER = "Adda ANN (published recipe)"
a = RESULTS[PAPER].pivot(index="repeat", columns="Output")
b = RESULTS[BEST].pivot(index="repeat", columns="Output")
tcrit = stats.t.ppf(0.975, N_REPEATS - 1)

c1 = []
for o in OUTPUTS + ["MEAN"]:
    if o == "MEAN":
        dR = (b["R"].mean(axis=1) - a["R"].mean(axis=1))
        pa_R, be_R = a["R"].mean(axis=1).mean(), b["R"].mean(axis=1).mean()
        red = lambda m: 100 * (1 - b[m].mean(axis=1).mean() / a[m].mean(axis=1).mean())
    else:
        dR = b["R"][o] - a["R"][o]
        pa_R, be_R = a["R"][o].mean(), b["R"][o].mean()
        red = lambda m, o=o: 100 * (1 - b[m][o].mean() / a[m][o].mean())
    ci = tcrit * dR.std(ddof=1) / np.sqrt(N_REPEATS)
    c1.append({"Output": o.replace("Permeate", ""),
               f"{PAPER}, held-out R": round(pa_R, 3), f"{BEST}, held-out R": round(be_R, 3),
               "delta R": f"{dR.mean():+.3f} +/- {ci:.3f}",
               "RMSE reduction": f"{red('RMSE'):+.0f}%",
               "MAE reduction":  f"{red('MAE'):+.0f}%",
               "AARD reduction": f"{red('AARD'):+.0f}%"})
C1 = pd.DataFrame(c1).set_index("Output")
C1.to_csv("compare_heldout.csv")
print("Table C1, out-of-fold, engineering units")
print(C1.to_string())
print("\ndelta R: paired difference per repeat; +/- = t(0.975, 4) * sd / sqrt(5), "
      "a repeat-to-repeat stability interval, not a confidence interval")


Table C1, out-of-fold, engineering units
              Adda ANN (published recipe), held-out R  Random forest, held-out R           delta R RMSE reduction MAE reduction AARD reduction
Output                                                                                                                                        
Conductivity                                    0.408                      0.737  +0.329 +/- 0.146           +43%          +38%           +38%
FlowRate                                        0.643                      0.818  +0.175 +/- 0.165           +40%          +26%           +26%
Recovery                                        0.687                      0.893  +0.206 +/- 0.103           +48%          +39%           +39%
MEAN                                            0.579                      0.816  +0.237 +/- 0.130           +43%          +38%           +35%

delta R: paired difference per repeat; +/- = t(0.975, 4) * sd / sqrt(5), a repeat-to-repeat stabilit

In [12]:
# Table C2: the paper's recipe scored in-sample and out-of-fold
paper_printed = {"PermeateConductivity": 0.969, "PermeateFlowRate": 0.942, "PermeateRecovery": 0.964}
pann_is = insample_R(lambda: PaperANN(lam=0.0, ensemble=False, max_nfev=3000))

C2 = pd.DataFrame([{
    "Output": o.replace("Permeate", ""),
    "paper printed R (in-sample)": paper_printed[o],
    "Adda ANN (reproduced), in-sample R":  round(pann_is[o], 3),
    "Adda ANN (reproduced), held-out R":   round(RESULTS[PAPER].groupby('Output').R.mean()[o], 3),
    "Improved ANN, held-out R":    round(RESULTS['Improved ANN (L2 + ensemble)'].groupby('Output').R.mean()[o], 3),
    f"{BEST}, held-out R":         round(RESULTS[BEST].groupby('Output').R.mean()[o], 3),
} for o in OUTPUTS]).set_index("Output")
C2.to_csv("compare_protocol.csv")
print("Table C2 (columns 1-2 in-sample, 3-5 out-of-fold)")
print(C2.to_string())


Table C2 (columns 1-2 in-sample, 3-5 out-of-fold)
              paper printed R (in-sample)  Adda ANN (reproduced), in-sample R  Adda ANN (reproduced), held-out R  Improved ANN, held-out R  Random forest, held-out R
Output                                                                                                                                                               
Conductivity                        0.969                               0.982                              0.408                     0.701                      0.737
FlowRate                            0.942                               0.950                              0.643                     0.776                      0.818
Recovery                            0.964                               0.965                              0.687                     0.828                      0.893


In [13]:
# Table C3: in-sample R on the 60-row split
rf_is  = insample_R(make_rf)
ann_is = insample_R(lambda: PaperANN(lam=ANN_LAMBDA, ensemble=True))
C3 = pd.DataFrame([{
    "Output": o.replace("Permeate", ""),
    "paper printed (in-sample)":   paper_printed[o],
    "Adda ANN (reproduced, in-sample)":   round(pann_is[o], 3),
    "Improved ANN (in-sample)":    round(ann_is[o], 3),
    f"{BEST} (in-sample)":         round(rf_is[o], 3),
} for o in OUTPUTS]).set_index("Output")
C3.to_csv("compare_insample.csv")
print("Table C3, in-sample R on the 60-row split")
print(C3.to_string())


Table C3, in-sample R on the 60-row split
              paper printed (in-sample)  Adda ANN (reproduced, in-sample)  Improved ANN (in-sample)  Random forest (in-sample)
Output                                                                                                                        
Conductivity                      0.969                             0.982                     0.958                      0.895
FlowRate                          0.942                             0.950                     0.910                      0.904
Recovery                          0.964                             0.965                     0.946                      0.949


The ± in C1 is t(0.975, 4)·sd/√5 over the 5 repeats. The repeats reuse the same 73 rows, so this is
a repeat-to-repeat stability interval and not a confidence interval (it is too narrow). The RMSE and
MAE changes in the MEAN row are ratios of means across outputs in different units and mostly follow
conductivity; AARD is the unit-free one.

## 6. Section D: prediction intervals

Gaussian process: mean ± 1.96σ. Quantile gradient boosting: separate models for the 2.5 % and
97.5 % quantiles. Coverage is measured out-of-fold; the target is 95 %.

In [14]:
def _interval_fold(method, tr, te):
    xs = StandardScaler().fit(X_all[tr])
    Xtr, Xte = xs.transform(X_all[tr]), xs.transform(X_all[te])
    lo = np.zeros((len(te), 3)); hi = np.zeros((len(te), 3)); mu = np.zeros((len(te), 3))
    for j in range(3):
        ys = StandardScaler().fit(Y_all[tr, j:j+1])
        ytr = ys.transform(Y_all[tr, j:j+1]).ravel()
        if method == "Gaussian process":
            g = GaussianProcessRegressor(
                    kernel=ConstantKernel() * Matern(nu=2.5, length_scale=1.0) + WhiteKernel(),
                    normalize_y=False, n_restarts_optimizer=2, random_state=0).fit(Xtr, ytr)
            m, s = g.predict(Xte, return_std=True)
            m = ys.inverse_transform(m.reshape(-1, 1)).ravel(); s = s * ys.scale_[0]
            lo[:, j], hi[:, j], mu[:, j] = m - 1.96 * s, m + 1.96 * s, m
        else:
            kw = dict(n_estimators=300, max_depth=2, learning_rate=0.05, subsample=0.9, random_state=0)
            ql = GradientBoostingRegressor(loss="quantile", alpha=0.025, **kw).fit(Xtr, ytr)
            qh = GradientBoostingRegressor(loss="quantile", alpha=0.975, **kw).fit(Xtr, ytr)
            l = ys.inverse_transform(ql.predict(Xte).reshape(-1, 1)).ravel()
            h = ys.inverse_transform(qh.predict(Xte).reshape(-1, 1)).ravel()
            lo[:, j], hi[:, j] = np.minimum(l, h), np.maximum(l, h); mu[:, j] = 0.5 * (l + h)
    return te, lo, hi, mu

splits = list(RepeatedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED).split(X_all))
cov_rows, rep0 = [], {}
for method in ["Gaussian process", "Quantile GBM"]:
    parts = Parallel(n_jobs=N_JOBS)(delayed(_interval_fold)(method, tr, te) for tr, te in splits)
    per_rep = {o: {"cov": [], "wid": []} for o in OUTPUTS}
    for rep in range(N_REPEATS):
        LO = np.full_like(Y_all, np.nan, float); HI = np.full_like(Y_all, np.nan, float)
        MU = np.full_like(Y_all, np.nan, float)
        for k in range(N_SPLITS):
            te, lo, hi, mu = parts[rep * N_SPLITS + k]
            LO[te], HI[te], MU[te] = lo, hi, mu
        for j, o in enumerate(OUTPUTS):
            inside = (Y_all[:, j] >= LO[:, j]) & (Y_all[:, j] <= HI[:, j])
            per_rep[o]["cov"].append(inside.mean())
            per_rep[o]["wid"].append((HI[:, j] - LO[:, j]).mean())
        if rep == 0:
            rep0[method] = (LO.copy(), HI.copy(), MU.copy())
    for o in OUTPUTS:
        cov_rows.append({"Output": o.replace("Permeate", ""), "Method": method,
                         "Coverage": np.mean(per_rep[o]["cov"]),
                         "Coverage_sd": np.std(per_rep[o]["cov"]),
                         "MeanWidth": np.mean(per_rep[o]["wid"]),
                         "unit": UNITS[o]})
cov = pd.DataFrame(cov_rows)
cov.to_csv("uncertainty_coverage.csv", index=False)
print("95% interval coverage (target 0.95), out-of-fold, mean over repeats")
print(cov.round(3).to_string(index=False))

95% interval coverage (target 0.95), out-of-fold, mean over repeats
      Output           Method  Coverage  Coverage_sd  MeanWidth  unit
Conductivity Gaussian process     0.907        0.005    139.043 uS/cm
    FlowRate Gaussian process     0.934        0.016      0.120  m3/h
    Recovery Gaussian process     0.907        0.016      2.809     %
Conductivity     Quantile GBM     0.742        0.037    124.575 uS/cm
    FlowRate     Quantile GBM     0.786        0.016      0.085  m3/h
    Recovery     Quantile GBM     0.786        0.021      2.098     %


In [15]:
plt.rcParams.update({"font.size": 9})
SHORT = {"PermeateConductivity": "Conductivity (µS/cm)", "PermeateFlowRate": "Flow rate (m³/h)",
         "PermeateRecovery": "Recovery (%)"}

lo, hi, mu = rep0["Gaussian process"]
fig, axes = plt.subplots(1, 3, figsize=(6.0, 2.5))
for j, (ax, o) in enumerate(zip(axes, OUTPUTS)):
    order = np.argsort(Y_all[:, j])
    x = np.arange(73)
    ax.fill_between(x, lo[order, j], hi[order, j], alpha=.25, color="#4a7fb5", lw=0, label="95% interval")
    ax.plot(x, mu[order, j], ".", color="#4a7fb5", ms=3, label="GP mean")
    ax.plot(x, Y_all[order, j], "x", color="k", ms=3, mew=.8, label="measured")
    c = cov[(cov.Method == "Gaussian process") & (cov.Output == o.replace("Permeate", ""))].Coverage.iloc[0]
    ax.set_title(f"{SHORT[o]}\ncoverage {c:.0%}", fontsize=9)
    ax.set_xlabel("rows, sorted")
    ax.set_xticks([])
    if j == 0:
        ax.legend(fontsize=9, loc="lower right", frameon=False, handlelength=1)
    ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig("uncertainty_intervals.png", dpi=300, bbox_inches="tight")
fig.savefig("uncertainty_intervals.pdf", bbox_inches="tight")
plt.show()

The GP intervals cover 90.7 %, 93.4 % and 90.7 % of rows, slightly below target. Quantile boosting
covers only 74-79 %, since the extreme quantiles cannot be estimated from about 65 training rows.
No calibration step is applied to either.

## 7. Section E: sensitivity

The paper applies Garson's method to one trained network. Here each input is permuted within the
held-out rows of every CV fold and the drop in R is recorded, for the random forest and the improved
ANN. The figure shows the random forest only.

In [16]:
def _perm_one_fold(make_est, X, Y, tr, te, n_in, n_shuffle, fseed):
    est = make_est().fit(X[tr], Y[tr])
    base = est.predict(X[te])
    baseR = np.array([np.corrcoef(Y[te, j], base[:, j])[0, 1] for j in range(3)])
    rng = np.random.default_rng(fseed)
    drop = np.zeros((n_in, 3))
    for i in range(n_in):
        acc = np.zeros(3)
        for _ in range(n_shuffle):
            Xp = X[te].copy(); Xp[:, i] = rng.permutation(Xp[:, i])
            P = est.predict(Xp)
            acc += baseR - np.array([np.corrcoef(Y[te, j], P[:, j])[0, 1] for j in range(3)])
        drop[i] = np.clip(acc / n_shuffle, 0, None)
    return drop                                   # (n_inputs, 3)

def cv_perm_importance(make_est, n_shuffle=20, tag=""):
    splits = list(RepeatedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS,
                                random_state=SEED).split(X_all))
    drops = Parallel(n_jobs=N_JOBS)(
        delayed(_perm_one_fold)(make_est, X_all, Y_all, tr, te, len(INPUTS), n_shuffle, 1000 + f)
        for f, (tr, te) in enumerate(splits))
    fold_imp = {o: [d[:, j] for d in drops] for j, o in enumerate(OUTPUTS)}
    rows = []
    for o in OUTPUTS:
        M = np.array(fold_imp[o])                 # (folds, inputs)
        share = 100 * M.mean(0) / M.mean(0).sum()
        rank1 = np.bincount(M.argmax(1), minlength=len(INPUTS))
        for i, inp in enumerate(INPUTS):
            rows.append({"Model": tag, "Output": o.replace("Permeate", ""), "Input": inp,
                         "PctShare": round(share[i], 1),
                         "CI_lo": round(np.percentile(M[:, i], 2.5), 3),
                         "CI_hi": round(np.percentile(M[:, i], 97.5), 3),
                         "Rank1_folds": int(rank1[i])})
    return pd.DataFrame(rows)

imp_rf  = cv_perm_importance(make_rf, tag="Random forest")
imp_ann = cv_perm_importance(lambda: PaperANN(lam=ANN_LAMBDA, ensemble=True), tag="Improved ANN")
imp = pd.concat([imp_rf, imp_ann], ignore_index=True)
imp.to_csv("permutation_importance_cv.csv", index=False)

for tag, d in [("Random forest", imp_rf), ("Improved ANN", imp_ann)]:
    print(f"\n{tag}  -  permutation importance (% share) with rank-1 fold count / {N_SPLITS*N_REPEATS}")
    print(d.pivot(index="Output", columns="Input", values="PctShare")[INPUTS].to_string())
    top = d.loc[d.groupby("Output").PctShare.idxmax(), ["Output", "Input", "Rank1_folds"]]
    print("  most important input:",
          ", ".join(f"{r.Output}->{r.Input} ({r.Rank1_folds}/{N_SPLITS*N_REPEATS})" for r in top.itertuples()))


Random forest  -  permutation importance (% share) with rank-1 fold count / 50
Input         Time  Pressure  Temperature  FeedFlow  ConductivityFeed
Output                                                               
Conductivity  28.1       3.4         64.9       0.6               3.0
FlowRate      63.9       1.0         24.1       1.0              10.0
Recovery      74.3       1.2         11.7       0.4              12.5
  most important input: Conductivity->Temperature (37/50), FlowRate->Time (33/50), Recovery->Time (40/50)

Improved ANN  -  permutation importance (% share) with rank-1 fold count / 50
Input         Time  Pressure  Temperature  FeedFlow  ConductivityFeed
Output                                                               
Conductivity  21.6      12.6         49.2       4.7              11.9
FlowRate      57.3       7.3         16.2      11.0               8.2
Recovery      40.9      11.0         22.2      15.5              10.4
  most important input: Conductivit

In [17]:
LABEL = {"Time": "Time", "Pressure": "Pressure", "Temperature": "Temperature",
         "FeedFlow": "Feed flow", "ConductivityFeed": "Feed conductivity"}
fig, axes = plt.subplots(1, 3, figsize=(6.0, 2.3), sharey=True)
for ax, o in zip(axes, OUTPUTS):
    d = imp_rf[imp_rf.Output == o.replace("Permeate", "")].set_index("Input").reindex(INPUTS[::-1])
    ax.barh([LABEL[i] for i in d.index], d.PctShare, color="#5aa06e", edgecolor="k", linewidth=.4)
    ax.set_title(SHORT[o].split(" (")[0], fontsize=9)
    ax.set_xlim(0, 100)
    ax.set_xlabel("share of R drop (%)"); ax.grid(axis="x", alpha=.3)
fig.tight_layout()
fig.savefig("permutation_importance_cv.png", dpi=300, bbox_inches="tight")
fig.savefig("permutation_importance_cv.pdf", bbox_inches="tight")
plt.show()

In [18]:
# Garson on the improved network, 12 seeds
def garson(net):
    W1, _, W2, _ = _ANNCore.unpack(net.nets_[0].t_)
    sh = np.abs(W1) / np.abs(W1).sum(1, keepdims=True)
    return pd.DataFrame({OUTPUTS[n]: 100 * (sh * np.abs(W2[n])[:, None]).sum(0)
                         / (sh * np.abs(W2[n])[:, None]).sum() for n in range(3)}, index=INPUTS).T

sx = (X_all.min(0), X_all.max(0), Y_all.min(0), Y_all.max(0))
gars = [garson(PaperANN(lam=ANN_LAMBDA, ensemble=False, seed=s).fit(X_all, Y_all))
        .loc["PermeateConductivity"] for s in range(12)]
G = pd.DataFrame(gars)
print("Garson importance for permeate conductivity, 12 random seeds, same data & model:")
print(G.describe().loc[["mean", "std", "min", "max"]].round(1).to_string())
print(f"\nTop input varies across seeds: {dict(pd.Series([g.idxmax() for g in gars]).value_counts())}")

Garson importance for permeate conductivity, 12 random seeds, same data & model:
      Time  Pressure  Temperature  FeedFlow  ConductivityFeed
mean  26.3      16.5         22.4       9.8              25.0
std    3.8       4.5          2.7       2.0               4.6
min   20.0      11.4         17.8       6.9              14.8
max   32.1      23.5         27.6      14.9              30.5

Top input varies across seeds: {'Time': np.int64(5), 'ConductivityFeed': np.int64(5), 'Temperature': np.int64(2)}


Both models give the same top input for each output: temperature for conductivity, time for flow rate
and recovery.

Caveats. Each fold has only 7-8 held-out rows, so the per-fold drops are noisy, and negative drops
are clipped to zero, which inflates the weak inputs. Time and temperature are correlated (r ≈ 0.93)
because the data are one continuous plant run, so the split of credit between them is not reliable.
Together they account for 93 % (random forest) and 71 % (improved ANN) of the conductivity
importance. Pressure is third at 3.4 % / 12.6 % and cannot be separated from feed conductivity.
The paper's conclusion that feed pressure is the most influential input (about 27 %) is not
supported.

Garson's method on the improved network, refitted with 12 seeds, picks time 5 times, feed
conductivity 5 times and temperature twice as the top input for conductivity.

## 8. Section F: out-of-fold parity plots

Repeat 0 only, so the R values differ from the 5-repeat means above.

In [19]:
oof_store = {}
for name, fac in [("Adda ANN (published recipe)", lambda: PaperANN(lam=0.0, ensemble=False, max_nfev=400, ext_scale=gx)),
                  ("Improved ANN", lambda: PaperANN(lam=ANN_LAMBDA, ensemble=True)),
                  (BEST, make_rf)]:
    oof = np.full_like(Y_all, np.nan, float)
    for k in range(N_SPLITS):
        tr, te = splits[k]                         # repeat 0
        oof[te] = fac().fit(X_all[tr], Y_all[tr]).predict(X_all[te])
    oof_store[name] = oof

ROW = {"Adda ANN (published recipe)": "Adda ANN", "Improved ANN": "Improved ANN", BEST: BEST}
fig, axes = plt.subplots(3, 3, figsize=(6.0, 6.4))
for r, name in enumerate(oof_store):
    for c, o in enumerate(OUTPUTS):
        ax = axes[r, c]
        y, p = Y_all[:, c], oof_store[name][:, c]
        lo_, hi_ = min(y.min(), p.min()), max(y.max(), p.max()); pad = .05 * (hi_ - lo_)
        ax.scatter(y, p, s=8, c="k")
        ax.plot([lo_ - pad, hi_ + pad], [lo_ - pad, hi_ + pad], "k:", lw=1)
        ax.set_xlim(lo_ - pad, hi_ + pad); ax.set_ylim(lo_ - pad, hi_ + pad)
        if r == 0:
            ax.set_title(SHORT[o], fontsize=9)
        if r == 2:
            ax.set_xlabel("Measured")
        if c == 0:
            ax.set_ylabel(f"{ROW[name]}\nPredicted")
        ax.text(.05, .86, f"R = {np.corrcoef(y, p)[0,1]:.3f}", transform=ax.transAxes)
fig.tight_layout()
fig.savefig("parity_oof.png", dpi=300, bbox_inches="tight")
fig.savefig("parity_oof.pdf", bbox_inches="tight")
plt.show()

## 9. Final model

The best model on the leaderboard is refitted on all 73 rows and saved with joblib. It takes the five
operating variables and returns the three permeate outputs in engineering units.

In [20]:
import joblib, re

FACTORY = {"MLR (paper baseline)": make_mlr, "Ridge": make_ridge,
           "Adda ANN (published recipe)": lambda: PaperANN(lam=0.0, ensemble=False, max_nfev=400, ext_scale=gx),
           "Improved ANN (L2 + ensemble)": lambda: PaperANN(lam=ANN_LAMBDA, ensemble=True),
           "SVR (RBF)": make_svr, "Gaussian process": make_gpr,
           "Random forest": make_rf, "XGBoost": make_xgb, "RF + XGB ensemble": RFXGB}

final_model = FACTORY[BEST]().fit(X_all, Y_all)
_slug = re.sub(r"[^a-z0-9]+", "_", BEST.lower()).strip("_")
MODEL_FILE = f"best_model_{_slug}.joblib"
joblib.dump(final_model, MODEL_FILE)

chosen = pd.DataFrame({
    "held-out R (10x5 CV)":  RESULTS[BEST].groupby("Output").R.mean().round(3),
    "held-out RMSE":         RESULTS[BEST].groupby("Output").RMSE.mean().round(3),
    "held-out MAE":          RESULTS[BEST].groupby("Output").MAE.mean().round(3),
    "held-out AARD %":       RESULTS[BEST].groupby("Output").AARD.mean().round(3),
    "Adda ANN held-out R":  RESULTS[PAPER].groupby("Output").R.mean().round(3),
    "paper printed R (in-sample)": pd.Series(pp),
}).reindex(OUTPUTS)
chosen.index = [o.replace("Permeate", "") for o in chosen.index]
chosen.loc["MEAN"] = chosen.mean()
chosen.to_csv("chosen_model_scorecard.csv")

print(f"chosen model: {BEST}")
print(f"saved -> {MODEL_FILE}  (load with joblib.load)")
print(chosen.round(3).to_string())
print(f"held-out R gain over the paper's ANN, same protocol: "
      f"{chosen.loc['MEAN','held-out R (10x5 CV)'] - chosen.loc['MEAN','Adda ANN held-out R']:+.3f} "
      f"(mean over the three outputs)")

# reload check
_check = joblib.load(MODEL_FILE).predict(X_all[:3])
print("\nsample predictions (first 3 rows), engineering units:")
print(pd.DataFrame(_check, columns=[o.replace("Permeate", "") for o in OUTPUTS]).round(2).to_string(index=False))

chosen model: Random forest
saved -> best_model_random_forest.joblib  (load with joblib.load)
              held-out R (10x5 CV)  held-out RMSE  held-out MAE  held-out AARD %  Adda ANN held-out R  paper printed R (in-sample)
Conductivity                 0.737         33.752        25.404            2.218                0.408                        0.969
FlowRate                     0.818          0.025         0.021            1.463                0.643                        0.942
Recovery                     0.893          0.489         0.389            0.879                0.687                        0.964
MEAN                         0.816         11.422         8.605            1.520                0.579                        0.958
held-out R gain over the paper's ANN, same protocol: +0.237 (mean over the three outputs)

sample predictions (first 3 rows), engineering units:
 Conductivity  FlowRate  Recovery
      1121.17      1.31     41.46
      1128.87      1.30     41.13
    

## Summary

- Out-of-fold, the paper's network scores mean R = 0.579, not the in-sample 0.969 / 0.942 / 0.964.
- A random forest reaches 0.816 on the same protocol (+0.237 R; AARD 35 % lower).
- With L2 weight decay and restart averaging the paper's own architecture reaches 0.768.
- GP prediction intervals cover 91-93 % of rows against a 95 % target.
- Time and temperature dominate the sensitivity analysis; pressure does not.
- Selection bias: the random forest, the ANN settings and the CV seed (42) were chosen or fixed on the
  same data that report the scores.